In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://endgbvnairobi.org/"

response = requests.get(url)

print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

print("Page title:", soup.title.get_text(strip=True) if soup.title else "Not found")

Status code: 200
Page title: Nairobi County GBV Directory | Home


**Finding the GBV related categories**

In [11]:
# Look for our selected categories in the homepage JavaScript

keywords = [
    "psychosocial",
    "shelter",
    "children",
    "focal",
    "sos"
]

for script in soup.find_all("script"):
    content = script.get_text()

    for keyword in keywords:
        position = content.lower().find(keyword)

        if position != -1:
            print(f"\n{keyword.upper()}")
            print(content[max(0, position - 150):position + 350])
            print("-" * 50)


PSYCHOSOCIAL
e-services.html",
    desc:"Hospitals, clinics and GBV treatment centres, including PEP and forensic care.",
    img:"images/mbagathi.png" },
  { id:"psychosocial", name:"Psychosocial Support", count:66, href:"category-pages/psychosocial-support.html",
    desc:"Counselling, mental health and emotional support services.",
    img:"images/Psychosocial.png" },
  { id:"legal", name:"Legal Services", count:58, href:"category-pages/legal-services.html",
    desc:"Legal aid, advocates, paralegals, chi
--------------------------------------------------

SHELTER
l", count:6,
    desc:"Police gender desks and survivor-focused response services across Nairobi County.",
    img:"images/genderdesk.png" },
  { id:"shelter", name:"Shelter & Security", count:38, href:"category-pages/shelter-services.html",
    desc:"Safe houses, rescue centres and secure accommodation.",
    img:"images/safehouse.png" },
  { id:"economic", name:"Economic Empowerment", count:26, href:"category-pages/econ

**Accessing the five specific directory categories**

In [12]:
# Selected GBV support categories
category_urls = {
    "Healthcare & GBV Clinics": "https://endgbvnairobi.org/category-pages/healthcare-services.html",
    "Psychosocial Support": "https://endgbvnairobi.org/category-pages/psychosocial-support.html",
    "Shelter & Security": "https://endgbvnairobi.org/category-pages/shelter-services.html",
    "Children's Officers": "https://endgbvnairobi.org/category-pages/childrens-officers.html",
    "GBV Focal Persons": "https://endgbvnairobi.org/category-pages/gbv-focal-persons.html"
}

# Check access to each category
for category, page_url in category_urls.items():

    try:
        response = requests.get(page_url, timeout=20)

        print(f"{category}: {response.status_code}")

    except requests.RequestException as e:
        print(f"{category}: Error - {e}")

Healthcare & GBV Clinics: 200
Psychosocial Support: 200
Shelter & Security: 200
Children's Officers: 200
GBV Focal Persons: 200


All five categories are accessible. 

Now determining how actual service provider records are stored.

**Inspecting the JavaScript on the category pages**

In [13]:
# Inspect JavaScript on each selected category page

for category, page_url in category_urls.items():

    response = requests.get(page_url, timeout=20)
    page_soup = BeautifulSoup(response.text, "html.parser")

    scripts = page_soup.find_all("script")

    print(f"\n{category}")
    print("Total scripts:", len(scripts))

    for i, script in enumerate(scripts):
        content = script.get_text(strip=True)

        if content:
            print(f"  Inline script {i + 1}: {len(content)} characters")

        elif script.get("src"):
            print(f"  External script: {script['src']}")

    print("-" * 50)


Healthcare & GBV Clinics
Total scripts: 2
  Inline script 1: 12687 characters
  External script: database-loader.js
--------------------------------------------------

Psychosocial Support
Total scripts: 2
  Inline script 1: 16072 characters
  External script: database-loader.js
--------------------------------------------------

Shelter & Security
Total scripts: 2
  Inline script 1: 10079 characters
  External script: database-loader.js
--------------------------------------------------

Children's Officers
Total scripts: 2
  Inline script 1: 6573 characters
  External script: database-loader.js
--------------------------------------------------

GBV Focal Persons
Total scripts: 2
  Inline script 1: 2985 characters
  External script: database-loader.js
--------------------------------------------------


All five category pages use the same external JavaScript file i.e: `database-loader.js` suggesting that the website may use a shared db to populate its service-provider listings.

**Inspecting the shared db loader**


In [14]:
from urllib.parse import urljoin

# Access the shared database loader
healthcare_url = category_urls["Healthcare & GBV Clinics"]

loader_url = urljoin(healthcare_url, "database-loader.js")

response = requests.get(loader_url, timeout=20)

print("Status code:", response.status_code)
print("Loader URL:", loader_url)
print("\nJavaScript preview:\n")
print(response.text[:4000])

Status code: 200
Loader URL: https://endgbvnairobi.org/category-pages/database-loader.js

JavaScript preview:

(() => {
  const categoryByPage = {
    'ambulance-services.html': 'ambulance',
    'childrens-officers.html': 'children',
    'economic-empowerment.html': 'economic',
    'gbv-focal-persons.html': 'focal',
    'healthcare-services.html': 'healthcare',
    'hotlines.html': 'hotlines',
    'legal-services.html': 'legal',
    'gender-desk.html': 'gender-desk',
    'mental-health-services.html': 'mental',
    'psychosocial-support.html': 'psychosocial',
    'refugee-services.html': 'refugee',
    'shelter-services.html': 'shelter'
  };
  const categoryId = categoryByPage[location.pathname.split('/').pop()];
  if (!categoryId) return;

  async function loadContacts() {
    const contacts = [];
    let offset = 0;
    while (true) {
      const response = await fetch(`/api/contacts?categoryId=${categoryId}&limit=250&offset=${offset}`);
      if (!response.ok) throw new Error('Direc

**Test the API with healthcare providers**

In [15]:
# API endpoint
api_url = "https://endgbvnairobi.org/api/contacts"

# Request healthcare service providers
params = {
    "categoryId": "healthcare",
    "limit": 250,
    "offset": 0
}

response = requests.get(api_url, params=params, timeout=20)

print("Status code:", response.status_code)

if response.ok:
    data = response.json()

    print("Response keys:", list(data.keys()))
    print("Number of records returned:", len(data.get("contacts", [])))

    # Examine one healthcare provider
    if data.get("contacts"):
        print("\nFirst healthcare provider:")
        print(data["contacts"][0])
else:
    print(response.text[:500])

Status code: 200
Response keys: ['categoryId', 'subcategoryId', 'status', 'limit', 'offset', 'contacts']
Number of records returned: 48

First healthcare provider:
{'id': 'd5dcbd14-7392-42a0-bc26-40a222f8e7af', 'categoryId': 'healthcare', 'categoryName': 'Healthcare Service Providers', 'subcategoryId': 'healthcare-hospital', 'subcategorySourceId': 'hospital', 'subcategoryName': 'Hospitals', 'name': 'Coptic Hospital (Hope Centre)', 'serviceType': 'Hospitals', 'location': 'Ngong Rd · Lunga Lunga · Industrial Area', 'phone': '0714220936, 0712024408, 0735-777058, 020 2043341', 'email': 'gsalano@coptichopecentre.org', 'status': 'needs_verification', 'displayOrder': 1, 'lastVerified': None, 'sourceLink': '', 'description': 'Comprehensive GBV care, PEP & forensic exam. Listed as a free service.'}


In [16]:
# Exploring the healthcare subcategories
# Convert healthcare records into a DataFrame
healthcare_df = pd.DataFrame(data["contacts"])

# Examine healthcare subcategories
print("Healthcare subcategories:\n")
print(healthcare_df["subcategoryName"].value_counts())

print("\nVerification status:\n")
print(healthcare_df["status"].value_counts(dropna=False))

print("\nSample providers by subcategory:\n")
print(
    healthcare_df[
        ["name", "subcategoryName", "serviceType", "description"]
    ].groupby("subcategoryName").head(3).to_string(index=False)
)

Healthcare subcategories:

subcategoryName
Community          25
Clinics            12
Hospitals           7
County / Police     3
SMS                 1
Name: count, dtype: int64

Verification status:

status
needs_verification    43
verified               5
Name: count, dtype: int64

Sample providers by subcategory:

                                                    name subcategoryName                                 serviceType                                                                                                                                                                      description
                           Coptic Hospital (Hope Centre)       Hospitals                                   Hospitals                                                                                                           Comprehensive GBV care, PEP & forensic exam. Listed as a free service.
                          Gertrude's Children's Hospital       Hospitals                    

# **Identify GBV-related healthcare providers**


In [ ]:
# Keywords for GBV-related healthcare services
gbv_keywords = (
    r"GBV|gender.based violence|sexual violence|"
    r"rape|survivor|PEP|post.exposure|"
    r"forensic|child protection|sexual assault"
)

# Search descriptions and service types
healthcare_df["gbv_related"] = (
    healthcare_df["description"].fillna("").str.contains(
        gbv_keywords, case=False, regex=True
    )
    |
    healthcare_df["serviceType"].fillna("").str.contains(
        gbv_keywords, case=False, regex=True
    )
)

# View potential GBV healthcare providers
gbv_healthcare = healthcare_df[
    healthcare_df["gbv_related"]
].copy()

print("Potential GBV-related healthcare providers:", len(gbv_healthcare))

print(
    gbv_healthcare[
        ["name", "subcategoryName", "description", "status"]
    ].to_string(index=False)
)

Potential GBV-related healthcare providers: 12
                                                    name subcategoryName                                                                                                                                                                      description             status
                           Coptic Hospital (Hope Centre)       Hospitals                                                                                                           Comprehensive GBV care, PEP & forensic exam. Listed as a free service. needs_verification
                          Gertrude's Children's Hospital       Hospitals                                                                                                                                         Paediatric emergency & child protection. needs_verification
                                          Mater Hospital       Hospitals                                                                          

**Classify healthcare records by support type**

In [18]:
# Classify the shortlisted healthcare providers
def classify_healthcare(row):

    text = (
        str(row["description"]) + " " +
        str(row["serviceType"])
    ).lower()

    if any(word in text for word in [
        "police", "sub-county gbv", "family planning"
    ]):
        return "Protection & Coordination"

    elif any(word in text for word in [
        "child protection", "women and children"
    ]):
        return "Child Protection & Healthcare"

    elif any(word in text for word in [
        "medical", "clinical", "hospital", "clinic",
        "pep", "forensic", "casualty"
    ]):
        return "GBV Medical Care"

    else:
        return "General GBV Support"


gbv_healthcare["support_type"] = gbv_healthcare.apply(
    classify_healthcare,
    axis=1
)

print(
    gbv_healthcare[
        ["name", "support_type", "status"]
    ].to_string(index=False)
)

                                                    name                  support_type             status
                           Coptic Hospital (Hope Centre)              GBV Medical Care needs_verification
                          Gertrude's Children's Hospital Child Protection & Healthcare needs_verification
                                          Mater Hospital              GBV Medical Care needs_verification
                                Mbagathi County Hospital              GBV Medical Care needs_verification
                                        Mutuini Hospital              GBV Medical Care needs_verification
                                          Tumaini Clinic              GBV Medical Care needs_verification
                  GVRC (Gender Violence Recovery Centre)              GBV Medical Care needs_verification
                                                USIKIMYE           General GBV Support needs_verification
                                  Wangu Kanja 

# **Extracting Psychosocial Support Providers**

In [19]:
# Retrieve psychosocial support providers
params = {
    "categoryId": "psychosocial",
    "limit": 250,
    "offset": 0
}

response = requests.get(api_url, params=params, timeout=20)
response.raise_for_status()

psychosocial_data = response.json()

# Convert to DataFrame
psychosocial_df = pd.DataFrame(psychosocial_data["contacts"])

print("Total psychosocial providers:", len(psychosocial_df))

print("\nSubcategories:")
print(psychosocial_df["subcategoryName"].value_counts())

print("\nVerification status:")
print(psychosocial_df["status"].value_counts(dropna=False))

print("\nSample providers:")
print(
    psychosocial_df[
        ["name", "serviceType", "description", "status"]
    ].head(10).to_string(index=False)
)

Total psychosocial providers: 91

Subcategories:
subcategoryName
Community              52
Counselling            20
Hospitals & clinics    12
Support groups          6
SMS                     1
Name: count, dtype: int64

Verification status:
status
needs_verification    85
verified               6
Name: count, dtype: int64

Sample providers:
                                       name serviceType                                                                  description             status
                               HESED Africa Counselling                       Health & social development. Listed as a free service.           verified
           Beacon of Hope (Rongai & others) Counselling                        Counselling across 6 areas. Listed as a free service. needs_verification
                             Beacon of Hope Counselling                                          Counselling & economic empowerment. needs_verification
                                      CREAW Cou

In [20]:
# Check duplicate organisation names
duplicate_names = psychosocial_df[
    psychosocial_df.duplicated(subset=["name"], keep=False)
].sort_values("name")

print("Number of records with duplicate names:", len(duplicate_names))

print("\nDuplicate organisations:")
print(
    duplicate_names[
        ["name", "subcategoryName", "location", "phone"]
    ].to_string(index=False)
)

Number of records with duplicate names: 2

Duplicate organisations:
                                    name subcategoryName location                    phone
Kenya Women and Children Wellness Centre     Counselling  Nairobi 0737-302963, 0703-302963
Kenya Women and Children Wellness Centre     Counselling                        0737302963


Removing the duplicate psychosocial record as both are under counselling and their phone numbers are the same only with different formats.

In [21]:
# Inspect both records before deduplication
psychosocial_df[
    psychosocial_df["name"] == "Kenya Women and Children Wellness Centre"
].T

,6,19
id,96549671-1d3f-4729-bc2b-f4b0ae25d6ff,3803877e-5983-4599-afe6-fdaf86572218
categoryId,psychosocial,psychosocial
categoryName,Psychosocial Support,Psychosocial Support
subcategoryId,psychosocial-counselling,psychosocial-counselling
subcategorySourceId,counselling,counselling
subcategoryName,Counselling,Counselling
name,Kenya Women and Children Wellness Centre,Kenya Women and Children Wellness Centre
serviceType,Counselling,
location,Nairobi,
phone,"0737-302963, 0703-302963",0737302963


In [22]:
# Identify the two duplicate records
duplicate_name = "Kenya Women and Children Wellness Centre"

# Keep the more complete record (index 6)
psychosocial_df = psychosocial_df.drop(index=19).copy()

# Preserve the other record's verification information
psychosocial_df.loc[6, "status"] = "needs_verification"
psychosocial_df.loc[6, "lastVerified"] = None

# Keep a reference to the verified source record
psychosocial_df.loc[6, "related_verified_record_id"] = (
    "3803877e-5983-4599-afe6-fdaf86572218"
)

# Confirm the result
print("Total psychosocial records:", len(psychosocial_df))

print(
    psychosocial_df.loc[
        psychosocial_df["name"] == duplicate_name,
        ["name", "phone", "email", "description",
         "status", "related_verified_record_id"]
    ].to_string(index=False)
)

Total psychosocial records: 90
                                    name                    phone          email                    description             status           related_verified_record_id
Kenya Women and Children Wellness Centre 0737-302963, 0703-302963 info@kwcwc.org Wellness & trauma counselling. needs_verification 3803877e-5983-4599-afe6-fdaf86572218
